# Stage 3 — instance segmentation

The canonical source algorithm is the only active segmentation path.

In [ ]:
from src.io import PipelinePaths

SAMPLE_ID = "44b6_0113de3b"
FRAME = 0
paths = PipelinePaths.discover()
sample_path = paths.sample_zarr(SAMPLE_ID)


In [ ]:
import matplotlib.pyplot as plt
from src.api import create_binary_mask, preprocess_volume, segment_instances
from src.io import load_timepoint

raw = load_timepoint(sample_path, FRAME)
preprocessed = preprocess_volume(raw)
binary_mask = create_binary_mask(preprocessed)
labels, trace = segment_instances(binary_mask, return_diagnostics=True)
print(trace.metrics)


In [ ]:
z = labels.shape[0] // 2
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].imshow(binary_mask[z], cmap="gray"); axes[0].set_title("component mask")
axes[1].imshow(trace.intermediates["markers"][z], cmap="nipy_spectral"); axes[1].set_title("markers")
axes[2].imshow(labels[z], cmap="nipy_spectral"); axes[2].set_title("instances")
plt.tight_layout()


In [ ]:
import pandas as pd
pd.DataFrame([
    {"component": d.subject_id, "outcome": d.outcome, **d.metrics}
    for d in trace.decisions
])
